# 🤖 Notebook 03 — Classification: Churn Prediction
### TeleConnect · 5 Classifiers: LR | DT | RF | SVM | KNN
> **Fully automated** — just run all cells (Runtime → Run all). No manual steps needed.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install',
    'pandas==2.1.4','numpy==1.26.2','scikit-learn==1.3.2',
    'matplotlib==3.8.2','seaborn==0.13.0','imbalanced-learn==0.11.0','-q'], check=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings, os, time, pickle
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi']=100
RANDOM_STATE=42; np.random.seed(RANDOM_STATE)
os.makedirs('reports/figures',exist_ok=True); os.makedirs('models',exist_ok=True)
print('✅ Setup complete')

In [ ]:
# ── Auto-load or rebuild data ─────────────────────────────────────────────────
def rebuild_data():
    DATA_URL='https://raw.githubusercontent.com/dsrscientist/dataset1/master/Telco-Customer-Churn.csv'
    df=pd.read_csv(DATA_URL)
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce').fillna(0)
    df['Churn']=(df['Churn']=='Yes').astype(int)
    service_cols=['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                  'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    df['ServiceCount']=df[service_cols].apply(lambda r:sum(v=='Yes' for v in r),axis=1)
    df['AvgMonthlySpend']=np.where(df['tenure']==0,df['MonthlyCharges'],df['TotalCharges']/df['tenure'])
    df['IsLongTermCustomer']=(df['tenure']>24).astype(int)
    le=LabelEncoder()
    for col in ['gender','Partner','Dependents','PhoneService','PaperlessBilling']:
        df[col]=le.fit_transform(df[col].astype(str))
    ohe=['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
         'DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaymentMethod']
    df=pd.get_dummies(df,columns=ohe,drop_first=True)
    bc=df.select_dtypes(include='bool').columns; df[bc]=df[bc].astype(int)
    df=df.drop(columns=['customerID'],errors='ignore')
    sc=[c for c in ['tenure','MonthlyCharges','TotalCharges','AvgMonthlySpend','ServiceCount'] if c in df.columns]
    ss=StandardScaler(); df[sc]=ss.fit_transform(df[sc])
    X=df.drop(columns=['Churn']); y=df['Churn']
    X_tr,X_tmp,y_tr,y_tmp=train_test_split(X,y,test_size=0.30,stratify=y,random_state=RANDOM_STATE)
    X_v,X_te,y_v,y_te=train_test_split(X_tmp,y_tmp,test_size=0.50,stratify=y_tmp,random_state=RANDOM_STATE)
    sm=SMOTE(random_state=RANDOM_STATE)
    X_tr,y_tr=sm.fit_resample(X_tr,y_tr)
    return X_tr,X_v,X_te,y_tr,y_v,y_te

try:
    X_train=pd.read_csv('data/processed/X_train.csv')
    X_val=pd.read_csv('data/processed/X_val.csv')
    X_test=pd.read_csv('data/processed/X_test.csv')
    y_train=pd.read_csv('data/processed/y_train.csv').squeeze()
    y_val=pd.read_csv('data/processed/y_val.csv').squeeze()
    y_test=pd.read_csv('data/processed/y_test.csv').squeeze()
    print('✅ Loaded preprocessed data')
except:
    print('Preprocessed files not found — rebuilding...')
    X_train,X_val,X_test,y_train,y_val,y_test=rebuild_data()
    print('✅ Data rebuilt')
print(f'Train:{X_train.shape} Val:{X_val.shape} Test:{X_test.shape}')

In [ ]:
# ── Evaluation helper ─────────────────────────────────────────────────────────
results_summary=[]
CV=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)

def evaluate(name,model,t):
    y_pred=model.predict(X_test)
    y_prob=model.predict_proba(X_test)[:,1] if hasattr(model,'predict_proba') else model.decision_function(X_test)
    acc=accuracy_score(y_test,y_pred)
    prec=precision_score(y_test,y_pred,zero_division=0)
    rec=recall_score(y_test,y_pred,zero_division=0)
    f1=f1_score(y_test,y_pred,zero_division=0)
    roc=roc_auc_score(y_test,y_prob)
    print(f'\n{"="*55}\n  {name}\n{"="*55}')
    print(f'  Accuracy:{acc:.4f}  Precision:{prec:.4f}  Recall:{rec:.4f}  F1:{f1:.4f}  ROC-AUC:{roc:.4f}  Time:{t:.1f}s')
    print(classification_report(y_test,y_pred,target_names=['Retained','Churned']))
    cm=confusion_matrix(y_test,y_pred)
    fig,ax=plt.subplots(figsize=(5,4))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,
                xticklabels=['Retained','Churned'],yticklabels=['Retained','Churned'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(f'Confusion Matrix — {name}')
    plt.tight_layout()
    sn=name.replace(' ','_').replace('(','').replace(')','').replace('/','_')
    plt.savefig(f'reports/figures/03_cm_{sn}.png',bbox_inches='tight'); plt.show()
    results_summary.append({'Model':name,'Accuracy':round(acc,4),'Precision':round(prec,4),
        'Recall':round(rec,4),'F1':round(f1,4),'ROC-AUC':round(roc,4),
        'Train Time(s)':round(t,2),'y_prob':y_prob,'y_test':y_test})
    return model
print('✅ Helpers ready')

In [ ]:
# ── Model 1: Logistic Regression ──────────────────────────────────────────────
print('Training Logistic Regression...')
t0=time.time()
lr=RandomizedSearchCV(LogisticRegression(class_weight='balanced',max_iter=1000,random_state=RANDOM_STATE),
    {'C':[0.01,0.1,1,10,100],'solver':['lbfgs','liblinear']},
    n_iter=8,cv=CV,scoring='roc_auc',random_state=RANDOM_STATE,n_jobs=-1)
lr.fit(X_train,y_train); lr_time=time.time()-t0
print(f'Best: {lr.best_params_}')
lr_best=lr.best_estimator_
evaluate('Logistic Regression',lr_best,lr_time)

In [ ]:
# ── Model 2: Decision Tree ────────────────────────────────────────────────────
print('Training Decision Tree...')
t0=time.time()
dt=RandomizedSearchCV(DecisionTreeClassifier(class_weight='balanced',random_state=RANDOM_STATE),
    {'max_depth':[3,5,7,10,None],'min_samples_split':[2,5,10,20],'min_samples_leaf':[1,2,5],'criterion':['gini','entropy']},
    n_iter=20,cv=CV,scoring='roc_auc',random_state=RANDOM_STATE,n_jobs=-1)
dt.fit(X_train,y_train); dt_time=time.time()-t0
print(f'Best: {dt.best_params_}')
dt_best=dt.best_estimator_
evaluate('Decision Tree',dt_best,dt_time)

In [ ]:
# ── Model 3: Random Forest ────────────────────────────────────────────────────
print('Training Random Forest...')
t0=time.time()
rf=RandomizedSearchCV(RandomForestClassifier(class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1),
    {'n_estimators':[100,200,300],'max_depth':[5,10,15,None],'min_samples_split':[2,5],'max_features':['sqrt','log2']},
    n_iter=15,cv=CV,scoring='roc_auc',random_state=RANDOM_STATE,n_jobs=-1)
rf.fit(X_train,y_train); rf_time=time.time()-t0
print(f'Best: {rf.best_params_}')
rf_best=rf.best_estimator_
evaluate('Random Forest',rf_best,rf_time)

In [ ]:
# ── Model 4: SVM ──────────────────────────────────────────────────────────────
print('Training SVM (may take a few minutes)...')
t0=time.time()
svm=RandomizedSearchCV(SVC(probability=True,class_weight='balanced',random_state=RANDOM_STATE),
    {'C':[0.1,1,10],'kernel':['rbf','linear'],'gamma':['scale','auto']},
    n_iter=8,cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE),
    scoring='roc_auc',random_state=RANDOM_STATE,n_jobs=-1)
svm.fit(X_train,y_train); svm_time=time.time()-t0
print(f'Best: {svm.best_params_}')
svm_best=svm.best_estimator_
evaluate('SVM',svm_best,svm_time)

In [ ]:
# ── Model 5: KNN ──────────────────────────────────────────────────────────────
print('Training KNN...')
t0=time.time()
knn=RandomizedSearchCV(KNeighborsClassifier(),
    {'n_neighbors':[3,5,7,9,11,15],'weights':['uniform','distance'],'metric':['euclidean','manhattan']},
    n_iter=15,cv=CV,scoring='roc_auc',random_state=RANDOM_STATE,n_jobs=-1)
knn.fit(X_train,y_train); knn_time=time.time()-t0
print(f'Best: {knn.best_params_}')
knn_best=knn.best_estimator_
evaluate('KNN',knn_best,knn_time)

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────────────
comp=pd.DataFrame([{k:v for k,v in r.items() if k not in['y_prob','y_test']} for r in results_summary]).set_index('Model')
print('\n'+'='*65+'\nFINAL COMPARISON TABLE\n'+'='*65)
print(comp.to_string())

metrics=['Accuracy','Precision','Recall','F1','ROC-AUC']
fig,axes=plt.subplots(1,5,figsize=(22,4))
for ax,m in zip(axes,metrics):
    vals=comp[m].sort_values()
    colors=['seagreen' if v==vals.max() else 'steelblue' for v in vals]
    vals.plot(kind='barh',ax=ax,color=colors,edgecolor='black')
    ax.set_title(m); ax.set_xlim(0,1)
    for i,(idx,v) in enumerate(vals.items()): ax.text(v+0.01,i,f'{v:.3f}',va='center',fontsize=8)
plt.suptitle('Model Comparison — All Metrics',fontsize=13,y=1.02)
plt.tight_layout()
plt.savefig('reports/figures/03_model_comparison.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
fig,ax=plt.subplots(figsize=(9,7))
colors=['steelblue','darkorange','seagreen','crimson','purple']
for r,color in zip(results_summary,colors):
    fpr,tpr,_=roc_curve(r['y_test'],r['y_prob'])
    ax.plot(fpr,tpr,lw=2,color=color,label=f"{r['Model']} (AUC={r['ROC-AUC']:.3f})")
ax.plot([0,1],[0,1],'k--',lw=1.5,label='Random')
ax.set_xlabel('False Positive Rate',fontsize=12); ax.set_ylabel('True Positive Rate',fontsize=12)
ax.set_title('ROC Curves — All Classification Models',fontsize=13)
ax.legend(loc='lower right'); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/03_roc_curves.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Best Model & Feature Importance ──────────────────────────────────────────
best_name=comp['ROC-AUC'].idxmax()
print(f'\n🏆 BEST MODEL: {best_name}')
print(comp.loc[best_name])
fi=pd.Series(rf_best.feature_importances_,index=X_train.columns)
top10=fi.sort_values(ascending=False).head(10)
fig,ax=plt.subplots(figsize=(10,5))
top10.sort_values().plot(kind='barh',ax=ax,color='seagreen',edgecolor='black')
ax.set_title(f'Top 10 Feature Importances — Random Forest',fontsize=13)
for i,(idx,v) in enumerate(top10.sort_values().items()): ax.text(v+0.001,i,f'{v:.4f}',va='center',fontsize=9)
plt.tight_layout()
plt.savefig('reports/figures/03_feature_importance.png',bbox_inches='tight'); plt.show()

# Save best model
with open('models/best_classifier.pkl','wb') as f: pickle.dump(rf_best,f)
comp.to_csv('reports/classification_comparison.csv')
print('✅ Best classifier saved!')